# RoomAnalysisModel Demo

This notebook checks that the static room model can be built from the current SVG fixtures without changing the existing matrix algorithms.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display

from features.scene_evaluator import evaluate_scene, scene_metric_overlay_svg
from features.scene_loader import load_structured_svg_scene


In [ ]:
fixtures = [
    Path('test/SVG_baseline.svg'),
    Path('test/SVG_oblique.svg'),
    Path('test/SVG_chaoticsvg.svg'),
    Path('test/SVG_potionNoRotation.svg'),
]

runs = []
for fixture in fixtures:
    scene = load_structured_svg_scene(fixture)
    matrix = evaluate_scene(scene)
    runs.append({'fixture': fixture.name, 'scene': scene, 'matrix': matrix, 'model': matrix['room_analysis_model']})


In [ ]:
layer_rows = []
for run in runs:
    model = run['model']
    layer_rows.append({
        'fixture': run['fixture'],
        'objects': len(model['object_layer']['objects']),
        'furniture_groups': len(model['object_layer']['furniture_groups']),
        'activity_regions': len(model['region_layer']['activity_regions']),
        'position_scopes': len(model['region_layer']['position_scopes']),
        'spatial_networks': len(model['network_layer']['spatial_networks']),
        'metric_records': len(model['metric_layer']['metrics']),
        'summaries': len(model['summary_layer']['summaries']),
    })

pd.DataFrame(layer_rows)


In [ ]:
metric_rows = []
for run in runs:
    counts = pd.DataFrame(run['model']['metric_layer']['metrics']).groupby(['family', 'target_layer']).size()
    for (family, target_layer), count in counts.items():
        metric_rows.append({'fixture': run['fixture'], 'family': family, 'target_layer': target_layer, 'count': count})

pd.DataFrame(metric_rows).sort_values(['fixture', 'family', 'target_layer'])


In [ ]:
summary_rows = []
for run in runs:
    for item in run['model']['summary_layer']['summaries']:
        summary_rows.append({'fixture': run['fixture'], **item})

pd.DataFrame(summary_rows)


## Baseline Visual Check

The overlay below is the existing metric overlay on top of the SVG base map. The new model itself is represented by the tables above.

In [ ]:
display(SVG(data=scene_metric_overlay_svg(runs[0]['scene'], runs[0]['matrix'])))
